# 1. FG-NET Dataset Setup
Canonical Task 03 notebook for **Identity-Preserving Face Age Progression for Missing Children**. Run `01_setup.ipynb` first, then run these cells in order in a runtime with that checkout and mounted Drive. This CPU-only workflow prepares metadata, splits, and longitudinal pairs. It downloads nothing, copies no images, and generates no faces or experimental results.

Obtain FG-NET separately according to applicable access/license terms and manually extract it into Drive. The filename convention below is an assumption that must be checked against your actual files: exactly three subject digits, `A` (case-insensitive), exactly two age digits, optionally one ASCII letter, then `.jpg`, `.jpeg`, or `.png` (case-insensitive). Examples: `001A05.jpg`, `001A05a.JPG`. Naming variations fail clearly rather than being guessed.


In [ ]:
from pathlib import Path
import sys
import json

PROJECT_ROOT = Path("/content/face-age-progression")
DRIVE_ROOT = Path("/content/drive/MyDrive/AI6132")
if not (PROJECT_ROOT / "configs/dataset.yaml").is_file():
    raise SystemExit("Run notebooks/01_setup.ipynb first and ensure the checkout includes Task 03.")
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))
from src.utils.config import load_config
from src.utils.paths import configure_paths
from src.data.fgnet import (discover_images, build_metadata, validate_metadata,
                           dataset_summary, write_prepared_dataset)
from src.data.splits import split_subjects, validate_splits
from src.data.pairs import build_pairs, validate_pairs
CONFIG = load_config(PROJECT_ROOT / "configs/dataset.yaml")


## 2. Dataset Paths
Change `RAW_ROOT` if extraction created a nested directory. Discovery is recursive. All CSV image paths are relative POSIX paths under this root; resolve them with `RAW_ROOT / image_path`. Keep raw files on Drive; no full-dataset copy to `/content` is needed.

Default output behavior refuses existing `metadata.csv`, `pairs.csv`, or `preparation.json`. Inspect previous outputs before explicitly setting `OVERWRITE = True`. Use only one writer per output root.


In [ ]:
PATHS = configure_paths("drive", project_root=PROJECT_ROOT, drive_root=DRIVE_ROOT)
RAW_ROOT = PATHS.storage / "data/fgnet/raw"
OUTPUT_ROOT = PATHS.storage / "data/fgnet/processed"
OVERWRITE = False
print("Manually extracted dataset:", RAW_ROOT)
print("Metadata output:", OUTPUT_ROOT)


## 3. Raw Dataset Validation
If data is absent, this cell prints placement instructions and every processing cell below skips work. After manually placing/extracting FG-NET, rerun from this section. Image discovery checks supported files and root containment; it does not decode pixels or verify face content.


In [ ]:
RAW_READY = False
try:
    images = discover_images(RAW_ROOT)
except ValueError as error:
    print(error)
    print(f"Place/extract FG-NET manually under {RAW_ROOT}, or update RAW_ROOT above.")
    print("Dataset processing stopped. No download or outputs will be created.")
else:
    RAW_READY = True
    print("Discovered image files:", len(images))


## 4. Metadata Construction
The reusable parser rejects malformed supported images. Metadata is ordered by `subject_id`, `age`, then `image_path`. Subject IDs preserve leading zeros as text. No raw files are changed. Verify the assumed filename convention against your real FG-NET files before relying on the resulting ages/identities.


In [ ]:
if RAW_READY:
    metadata = build_metadata(RAW_ROOT)
    print("Metadata records:", len(metadata))
else:
    print("Skipped: raw dataset is not ready.")


## 5. Subject-Disjoint Split
Unique subject IDs are sorted and shuffled with a local RNG using the explicit configured seed. Default subject ratios are `train=0.7`, `val=0.15`, `test=0.15`. Largest-remainder allocation rounds subject counts, with ties in train/val/test order; very small datasets can have empty splits. All images and ages of a subject remain in one split. A single metadata table preserves assignment in a `split` column.


In [ ]:
if RAW_READY:
    metadata = split_subjects(metadata, **CONFIG["split"])
    print("Split settings:", CONFIG["split"])


## 6. Longitudinal Pair Construction
Subject-disjoint splitting and within-subject longitudinal pairing are separate operations. All valid younger-to-older image combinations are generated by default, including within train/val/test separately. `target_age > source_age`; equal-age pairs are excluded. Configure source/target age bounds and age-gap bounds in `configs/dataset.yaml`. An optional per-subject cap keeps the first K pairs in source-age/path then target-age/path order; it is deterministic and does not sample randomly. With no cap, output can grow quadratically per subject.

Pair age-gap labels: 1–5 → `0-5`, 6–10 → `6-10`, 11+ → `11+`. These are planned subgroup labels, not performance metrics.


In [ ]:
if RAW_READY:
    pairs = build_pairs(metadata, **CONFIG["pairs"])
    print("Pair settings:", CONFIG["pairs"])
    print("Longitudinal pairs:", len(pairs))


## 7. Leakage and Integrity Checks
Validation fails on metadata duplicates, invalid ages, missing paths, split subject overlap, cross-subject pair endpoints, incorrect pair ages/gaps, or inconsistent split/group labels. Serious problems are never silently repaired.


In [ ]:
if RAW_READY:
    validate_metadata(metadata, raw_root=RAW_ROOT, check_paths=True)
    validate_splits(metadata)
    validate_pairs(pairs, metadata)
    print("Metadata paths, subject-disjoint splits, and longitudinal pair integrity passed.")


## 8. Dataset Summary
Counts and age ranges below are computed from the available data. No expected FG-NET statistics are hard-coded. Metadata validation cannot establish image authenticity or dataset completeness.


In [ ]:
if RAW_READY:
    summary = dataset_summary(metadata, pairs)
    print(json.dumps(summary, indent=2, sort_keys=True))


## 9. Output Files
Writes `metadata.csv` (`subject_id, age, image_path, split`), `pairs.csv` (subject, endpoints, ages, gap, group, split), and `preparation.json` (raw root, effective settings, summary). Files are staged and each is replaced atomically on filesystems supporting rename; the bundle is not one transaction. An interrupted write may leave a partial bundle: inspect it, then rerun with explicit overwrite. Mounted Drive persistence/rename behavior still requires real Colab validation. Raw data and unrelated output filenames are preserved.


In [ ]:
if RAW_READY:
    write_prepared_dataset(RAW_ROOT, OUTPUT_ROOT, metadata, pairs,
                           dict(dataset=CONFIG["dataset"], split=CONFIG["split"],
                                pairs=CONFIG["pairs"]), overwrite=OVERWRITE)
    for name in ("metadata.csv", "pairs.csv", "preparation.json"):
        print(OUTPUT_ROOT / name)
else:
    print("No outputs written: place the raw dataset first.")


## 10. Next Steps
Review filename parsing and the summary against the actual FG-NET extraction. Keep all raw/processed datasets in Drive and never commit them to GitHub. When reading CSVs, preserve subject IDs as strings and convert ages/gaps to integers. Task 03 ends here: M0, M1, M2 and experimental evaluation remain unimplemented. Future generation concerns plausible visual candidates, not predictions of a specific child's actual future appearance.

After a Colab disconnection, rerun setup and this workflow; persistent data stays on Drive. Do not infer real-data correctness from synthetic Codex tests. This notebook does not require CUDA, including on Colab Free.
